In [11]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import h5py
import re

root = "/home/lwu/work/NarrativeGenomics/region_narration/code/pipeline/ARCHS4_LW/refining/data_slash_results/"
filename = root + 'archs4_gene_human_v2.1_normalized_LW20230126.h5'
with h5py.File(filename, 'r') as f:
    low_cat_matrix = f['data/lowest_tier_category_matrix2'][:]
    high_cat_matrix = f['data/highest_tier_category_matrix2'][:] 
    gene_names = f['meta/genes/all_gene_names'][:]
    gene_description = f['meta/genes/gene_description'][:]
    category_names = f['meta/biosamples/category'][:]

category_names = [x.decode('UTF-8') for x in category_names]
gene_names = [x.decode('UTF-8') for x in gene_names]
gene_description = [x.decode('UTF-8') for x in gene_description]

category_list  = ["Heart", "Atrium", "Cardiac Muscle Fiber", "Pericardium", "Valve", "Ventricle","HUVEC", 
    "Adipose", "Adipocyte", "Myofibroblast", "Osteoblast", "Stromal Cell", 
    "MG63", "K562", "Bone Marrow", "Chondrocyte", "HT1080", "NHDF",
    "HCT116", "HT29", "RKO", "Colon", "Colonic Muscosa", "Intestinal Epithelial Cell", 
    "Small Intestine", "Liver", "Hepatic Stellate Cell", "Hepatocyte", "Primary Liver Cell",
    "HEPG2", "Alpha Cell", "Beta Cell", "Pancreatic Islet", 
    "GSE81547", "PANC1", "Gastric Epithelial Cell", "Esophagus","Granulocyte", "Neutrophil", "Blymphocyte", 
    "Plasma Cell", "Plasmacytoid Dendritic Cell", 
    "Tlymphocyte", "GSM2679484", "JURKAT", "MT4", "NALM6", "REH", "RPMI8226", "Alveolar", 
    "Dendritic", "Kupffer", "Macrophage", "Macroglia",  "THP1", "U937", "JEG3", "Thymus", 
    "Thymocyte", "Spleen",
    "Skin", "Basal Cell", "Fibroblast", "Hair Follicle", "Keratinocyte", "Melanocyte",
    "A431", "HACAT", "HNSCC","Skeletal Muscle", "Myoblast", "Airway Smooth Muscle", "Respiratory Smooth Muscle", 
    "Vascular Smooth Muscle", "LHCNM2","Brain", "DAOY", "IMR32", "SKNSH", "Astrocyte", "Cerebellum", "Cerebral Cortex", 
    "Hypothalamus", "Medula", "Midbrain", "Neuron", "Oligodendrocyte", "Pons", "Spinal Cord", 
    "Thalamus", "Retina", "Motor Neuron", "Sensory Neuron","Lung", "Lung Epithelial", "A549", "H1299", "IMR90", "MRC5", "NCIH460", "NHBE", 
    "Trachea","Breast", "Mammary Gland", "MCF7", "MCF10", "MDAMB231", "MDAMB468", "SKBR3", "T47D", 
    "HELA", "Kidney", "Podocyte", "Renal Cortex", "293F", "FLPIN TREX 293", "HEK293", "C42", 
    "DU145", "LNCAP", "PC3", "Ovary", "Granulosa", "Oocyte", "ES2", "SKOV3", "Testis"]

category_dict  = {
    "Cardiovascular System" : ["Heart", "Atrium", "Cardiac Muscle Fiber", 
                               "Pericardium", "Valve", "Ventricle",
                               "HUVEC"], 
    
    "Connective Tissue" : ["Adipose", "Adipocyte", "Myofibroblast", "Osteoblast", "Stromal Cell", 
                           "MG63", "K562", "Bone Marrow", "Chondrocyte", "HT1080", "NHDF"],

    "Digestive System": ["HCT116", "HT29", "RKO", "Colon", "Colonic Muscosa", "Intestinal Epithelial Cell", 
                         "Small Intestine", "Liver", "Hepatic Stellate Cell", "Hepatocyte", "Primary Liver Cell",
                         "HEPG2", "Alpha Cell", "Beta Cell", "Pancreatic Islet", 
                         "GSE81547", "PANC1", "Gastric Epithelial Cell", "Esophagus"],
    
    "Immune System" : ["Granulocyte", "Neutrophil", "Blymphocyte", "Plasma Cell", "Plasmacytoid Dendritic Cell", 
                       "Tlymphocyte", "GSM2679484", "JURKAT", "MT4", "NALM6", "REH", "RPMI8226", "Alveolar", 
                       "Dendritic", "Kupffer", "Macrophage", "Macroglia",  "THP1", "U937", "JEG3", "Thymus", 
                       "Thymocyte", "Spleen"],
    
    "Integumentary System" : ["Skin", "Basal Cell", "Fibroblast", "Hair Follicle", "Keratinocyte", "Melanocyte",
                              "A431", "HACAT", "HNSCC"], 
    
    "Muscular System" : ["Skeletal Muscle", "Myoblast", "Airway Smooth Muscle", "Respiratory Smooth Muscle", 
                         "Vascular Smooth Muscle", "LHCNM2"], 
    
    "Nervous System" : ["Brain", "DAOY", "IMR32", "SKNSH", "Astrocyte", "Cerebellum", "Cerebral Cortex", 
                        "Hypothalamus", "Medula", "Midbrain", "Neuron", "Oligodendrocyte", "Pons", "Spinal Cord", 
                        "Thalamus", "Retina", "Motor Neuron", "Sensory Neuron"], 
    
    "Respiratory System" : ["Lung", "Lung Epithelial", "A549", "H1299", "IMR90", "MRC5", "NCIH460", "NHBE", 
                            "Trachea"], 
    
    "Urogenital System" : ["Breast", "Mammary Gland", "MCF7", "MCF10", "MDAMB231", "MDAMB468", "SKBR3", "T47D", 
                           "HELA", "Kidney", "Podocyte", "Renal Cortex", "293F", "FLPIN TREX 293", "HEK293", "C42", 
                           "DU145", "LNCAP", "PC3", "Ovary", "Granulosa", "Oocyte", "ES2", "SKOV3", "Testis"]
}

In [34]:
low_cat_matrix[0][0:3]

array([0.05616796, 0.04883453,        nan])

In [35]:
high_cat_matrix[0][0:3]

array([0.03641399, 0.12006714, 0.4371865 ])

In [12]:
from IPython.display import display

df = pd.DataFrame(low_cat_matrix, index=gene_names)
df.columns = category_list
#display(df)

,Heart,Atrium,Cardiac Muscle Fiber,Pericardium,Valve,Ventricle,HUVEC,Adipose,Adipocyte,Myofibroblast,...,C42,DU145,LNCAP,PC3,Ovary,Granulosa,Oocyte,ES2,SKOV3,Testis
A1BG,0.056168,0.048835,NaN,0.020615,0.037192,0.018626,0.035549,0.054395,0.080823,NaN,...,0.113614,0.042563,0.079865,0.021164,0.084695,0.161398,0.040065,0.056893,0.113216,0.018528
A1BG-AS1,0.032927,0.046450,NaN,0.022886,0.064558,0.027351,0.055125,0.046482,0.084185,NaN,...,0.114939,0.067360,0.126244,0.033287,0.054042,0.132141,0.091088,0.056802,0.102860,0.015722
A1CF,0.002484,0.001940,NaN,0.002102,0.002034,0.001614,0.001996,0.001682,0.010347,NaN,...,0.041208,0.002674,0.002804,0.009441,0.005889,0.003026,0.199834,0.011093,0.002924,0.001013
A2M,1.764354,2.772235,NaN,2.454062,1.057593,2.514335,0.156536,2.062961,0.536379,NaN,...,0.072369,0.006418,0.007191,0.009785,0.655399,0.127424,0.004551,0.281067,0.110783,0.882434
A2M-AS1,0.099394,0.099398,NaN,0.039510,0.034832,0.112690,0.019118,0.088403,0.036852,NaN,...,0.017734,0.030222,0.034285,0.013087,0.030193,0.008025,0.001430,0.017735,0.012415,0.015684
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZYG11B,0.417928,0.375017,NaN,0.209064,0.331834,0.446162,0.224431,0.302620,0.391044,NaN,...,0.212086,0.156977,0.203440,0.182462,0.230204,0.170460,0.030033,0.193103,0.228547,0.293389
ZYX,1.022740,0.987605,NaN,1.620014,1.468715,0.829994,2.160126,1.132355,1.446744,NaN,...,0.663281,1.440293,0.574266,2.263474,1.348975,2.201959,0.365838,1.093452,1.111386,0.851703
ZYXP1,0.000449,0.000213,NaN,0.001748,0.000394,0.000339,0.000638,0.000384,0.000664,NaN,...,0.000318,0.000498,0.000355,0.000569,0.000767,0.000801,0.002038,0.000401,0.000761,0.000406
ZZEF1,0.255312,0.225258,NaN,0.295227,0.243531,0.208332,0.244360,0.228001,0.218224,NaN,...,0.219436,0.195298,0.208366,0.159518,0.180144,0.179949,0.052299,0.157271,0.168155,0.195212


---
# Calculations

In [36]:

# Compute row-wise averages while skipping NaN values
row_averages = np.nanmean(low_cat_matrix, axis=1)

# Create new matrix where each value is divided by row average
low_cat_oe_matrix = np.divide(low_cat_matrix, row_averages[:, np.newaxis], 
                       out=np.zeros_like(low_cat_matrix), where=row_averages[:, np.newaxis]!=0)

print("Low Cat matrix shape:")
print(low_cat_matrix.shape)
print(low_cat_matrix[0][:5])
print()
print("Average:", row_averages[0])
print()
print("Low Cat OE matrix shape:")
print(low_cat_oe_matrix.shape)
print(low_cat_oe_matrix[0][:5])

Low Cat matrix shape:
(57447, 127)
[0.05616796 0.04883453        nan 0.02061483 0.03719162]

Average: 0.09820224169868699

Low Cat OE matrix shape:
(57447, 127)
[0.57196205 0.49728533        nan 0.20992223 0.37872471]


In [37]:
low_cat_log2_oe_matrix = np.log2(low_cat_oe_matrix)

In [22]:
from IPython.display import display

df = pd.DataFrame(low_cat_log2_oe_matrix, index=gene_names)
df.columns = category_list
#display(df)

---
# Results

In [23]:
df = pd.DataFrame(low_cat_matrix, index=gene_names)
df.columns = category_list

for col in df.columns:
    top_10 = df[col].nlargest(10)
    print('Top 10 values in Column "{}":'.format(col))
    print(top_10)
    print()
    print('Gene descriptions for Column "{}":'.format(col))
    
    for gene in top_10.index:
        index = gene_names.index(gene)
        print(gene_description[index])
    print()


Top 10 values in Column "Heart":
TPM1               5.742830
TNNT2              5.338282
ACTC1              5.300183
AHNAK              5.196548
GAPDH              5.145582
RPL41              5.069228
TNNC1              4.838087
ENSG00000283907    4.796245
MYH7               4.612605
MYL2               4.601054
Name: Heart, dtype: float64

Gene descriptions for Column "Heart":
tropomyosin 1
troponin T2, cardiac type
actin alpha cardiac muscle 1
AHNAK nucleoprotein
glyceraldehyde-3-phosphate dehydrogenase
ribosomal protein L41
troponin C1, slow skeletal and cardiac type
novel transcript, antisense to ATP4A
myosin heavy chain 7
myosin light chain 2

Top 10 values in Column "Atrium":
NPPA      8.230254
MYL7      7.551266
ACTC1     6.878828
MYH6      6.852111
TNNT2     6.524649
TPM1      6.114578
MYL4      6.088424
DES       5.944048
ANKRD1    5.900489
CRYAB     5.670695
Name: Atrium, dtype: float64

Gene descriptions for Column "Atrium":
natriuretic peptide A
myosin light chain 7
actin al

Top 10 values in Column "Small Intestine":
RN7SL1             7.481959
RN7SL2             7.057652
AHNAK              5.908737
B2M                5.805965
SNORD3D            5.525371
ENSG00000283907    5.473805
RPL41              5.406175
TMSB4X             5.384495
EEF1A1             5.290359
RPS18              4.897919
Name: Small Intestine, dtype: float64

Gene descriptions for Column "Small Intestine":
RNA component of signal recognition particle 7SL1
RNA component of signal recognition particle 7SL2
AHNAK nucleoprotein
beta-2-microglobulin
small nucleolar RNA, C/D box 3D
novel transcript, antisense to ATP4A
ribosomal protein L41
thymosin beta 4 X-linked
eukaryotic translation elongation factor 1 alpha 1
ribosomal protein S18

Top 10 values in Column "Liver":
ALB                6.019595
FTL                6.013129
HP                 5.001778
RPL41              4.953933
SERPINA1           4.902178
ENSG00000283907    4.858437
APOA1              4.730877
APOA2              4.667427
FT

Top 10 values in Column "Alveolar":
FTL                7.357284
FTH1               6.589206
TMSB4X             6.088045
B2M                5.949586
AHNAK              5.144226
TMSB10             5.028673
ENSG00000283907    5.009946
EEF1A1             4.985978
RPL41              4.948868
ACTB               4.946105
Name: Alveolar, dtype: float64

Gene descriptions for Column "Alveolar":
ferritin light chain
ferritin heavy chain 1
thymosin beta 4 X-linked
beta-2-microglobulin
AHNAK nucleoprotein
thymosin beta 10
novel transcript, antisense to ATP4A
eukaryotic translation elongation factor 1 alpha 1
ribosomal protein L41
actin beta

Top 10 values in Column "Dendritic":
FTL        6.837067
TMSB4X     6.667645
B2M        6.179454
HLA-DRA    5.934298
FTH1       5.883748
CD74       5.827270
TMSB10     5.621153
ACTB       5.504899
RPL41      5.383096
AHNAK      5.238897
Name: Dendritic, dtype: float64

Gene descriptions for Column "Dendritic":
ferritin light chain
thymosin beta 4 X-linked
beta

Top 10 values in Column "Vascular Smooth Muscle":
GAPDH     6.441466
FTL       6.215082
COL1A1    6.074400
RPL41     5.999033
TMSB4X    5.927418
FTH1      5.860288
VIM       5.816866
AHNAK     5.767804
TMSB10    5.726166
COL1A2    5.722030
Name: Vascular Smooth Muscle, dtype: float64

Gene descriptions for Column "Vascular Smooth Muscle":
glyceraldehyde-3-phosphate dehydrogenase
ferritin light chain
collagen type I alpha 1 chain
ribosomal protein L41
thymosin beta 4 X-linked
ferritin heavy chain 1
vimentin
AHNAK nucleoprotein
thymosin beta 10
collagen type I alpha 2 chain

Top 10 values in Column "LHCNM2":
GAPDH     7.142394
FTH1      6.846385
TMSB10    6.546708
FTL       6.333931
AHNAK     6.312670
MT2A      6.290889
B2M       6.100877
S100A6    5.969637
VIM       5.926932
EEF1A1    5.783252
Name: LHCNM2, dtype: float64

Gene descriptions for Column "LHCNM2":
glyceraldehyde-3-phosphate dehydrogenase
ferritin heavy chain 1
thymosin beta 10
ferritin light chain
AHNAK nucleoprotein
metal

Top 10 values in Column "H1299":
GAPDH     7.544103
RPL41     6.773821
EEF1A1    6.695163
RPS18     6.004704
TMSB10    5.944571
RPS19     5.777851
RPS12     5.595477
RPS16     5.545075
RPL13     5.492190
RPL8      5.396575
Name: H1299, dtype: float64

Gene descriptions for Column "H1299":
glyceraldehyde-3-phosphate dehydrogenase
ribosomal protein L41
eukaryotic translation elongation factor 1 alpha 1
ribosomal protein S18
thymosin beta 10
ribosomal protein S19
ribosomal protein S12
ribosomal protein S16
ribosomal protein L13
ribosomal protein L8

Top 10 values in Column "IMR90":
GAPDH     6.765511
ACTB      6.253539
TMSB10    6.154713
FTL       5.927146
VIM       5.829569
LGALS1    5.815059
AHNAK     5.772658
TMSB4X    5.525071
ACTG1     5.474600
EEF1A1    5.469429
Name: IMR90, dtype: float64

Gene descriptions for Column "IMR90":
glyceraldehyde-3-phosphate dehydrogenase
actin beta
thymosin beta 10
ferritin light chain
vimentin
galectin 1
AHNAK nucleoprotein
thymosin beta 4 X-linked
ac

Top 10 values in Column "DU145":
GAPDH     7.078236
RPL41     6.715854
RPS18     6.501613
FTH1      6.177126
FTL       5.793427
S100A6    5.775132
AHNAK     5.622136
ENO1      5.559997
RPS2      5.508133
ACTB      5.499326
Name: DU145, dtype: float64

Gene descriptions for Column "DU145":
glyceraldehyde-3-phosphate dehydrogenase
ribosomal protein L41
ribosomal protein S18
ferritin heavy chain 1
ferritin light chain
S100 calcium binding protein A6
AHNAK nucleoprotein
enolase 1
ribosomal protein S2
actin beta

Top 10 values in Column "LNCAP":
RPL41     7.407375
RPS18     6.742820
RPS2      6.239265
GAPDH     6.095460
RPL13A    5.944676
RPL7A     5.892479
EEF1A1    5.771103
RPS24     5.600378
RPLP0     5.556416
RPL26     5.507384
Name: LNCAP, dtype: float64

Gene descriptions for Column "LNCAP":
ribosomal protein L41
ribosomal protein S18
ribosomal protein S2
glyceraldehyde-3-phosphate dehydrogenase
ribosomal protein L13a
ribosomal protein L7a
eukaryotic translation elongation factor 1 al

In [24]:
df = pd.DataFrame(low_cat_oe_matrix, index=gene_names)
df.columns = category_list

for col in df.columns:
    top_10 = df[col].nlargest(10)
    print('Top 10 values in Column "{}":'.format(col))
    print(top_10)
    print()
    print('Gene descriptions for Column "{}":'.format(col))
    
    for gene in top_10.index:
        index = gene_names.index(gene)
        print(gene_description[index])
    print()


Top 10 values in Column "Heart":
RN7SKP288          60.015807
CACNA1C-IT1        59.236888
ENSG00000286639    49.689579
ENSG00000267784    49.565267
BTF3L4P4           45.925513
ENSG00000232056    45.245420
ENSG00000238156    43.591272
ENSG00000250500    42.585218
ENSG00000271401    37.209561
ENSG00000237324    36.547886
Name: Heart, dtype: float64

Gene descriptions for Column "Heart":
RN7SK pseudogene 288
CACNA1C intronic transcript 1
novel transcript
novel transcript, antisense to titin
basic transcription factor 3 like 4 pseudogene 4
novel transcript
novel transcript
eukaryotic translation initiation factor 2, subunit 3 gamma, 52kDa (EIF2S3) pseudogene
novel transcript
novel transcript

Top 10 values in Column "Atrium":
BMP10              92.512588
MIR208A            85.817873
ENSG00000287891    67.385660
MYBPHL             66.685877
ENSG00000261606    66.180544
ENSG00000225519    64.210066
LMAN1L             60.449920
MYH6               57.154627
LINC01880          54.779973
LINC0

Top 10 values in Column "Hepatic Stellate Cell":
ENSG00000284695    23.592698
MIR656             21.417322
SULT1E1            18.859414
MIR412             18.293262
MIR369             16.415354
MIR410             14.748502
MIR409             13.983665
MIR377             12.240145
ENSG00000228707    11.101532
COLEC11            10.900319
Name: Hepatic Stellate Cell, dtype: float64

Gene descriptions for Column "Hepatic Stellate Cell":
novel protein
microRNA 656
sulfotransferase family 1E member 1
microRNA 412
microRNA 369
microRNA 410
microRNA 409
microRNA 377
novel transcript
collectin subfamily member 11

Top 10 values in Column "Hepatocyte":
GAGE12C    48.815310
GAGE12D    48.815310
APCS       44.503269
HPX        39.852415
HAO1       38.711034
HABP2      38.706938
FGB        37.120937
ANGPTL3    36.711506
FGA        36.671618
LBP        36.518493
Name: Hepatocyte, dtype: float64

Gene descriptions for Column "Hepatocyte":
G antigen 12C
G antigen 12D
amyloid P component, serum
hemope

Top 10 values in Column "Macrophage":
ACOD1              58.880479
ENSG00000225107    37.620318
CHIT1              30.397937
ADAMDEC1           29.455373
LINC01181          27.816142
ETV3L              27.277054
DCSTAMP            25.108518
GPR31              25.014187
ENSG00000270980    24.960766
LINC02244          24.752556
Name: Macrophage, dtype: float64

Gene descriptions for Column "Macrophage":
aconitate decarboxylase 1
novel transcript
chitinase 1
ADAM like decysin 1
long intergenic non-protein coding RNA 1181
ETS variant transcription factor 3 like
dendrocyte expressed seven transmembrane protein
G protein-coupled receptor 31
coiled-coil-helix-coiled-coil-helix domain containing 2 (CHCHD2) pseudogene
long intergenic non-protein coding RNA 2244

Top 10 values in Column "Macroglia":
A1BG        NaN
A1BG-AS1    NaN
A1CF        NaN
A2M         NaN
A2M-AS1     NaN
A2ML1       NaN
A2ML1-AS1   NaN
A2ML1-AS2   NaN
A2MP1       NaN
A3GALT2     NaN
Name: Macroglia, dtype: float64

Gene d

Top 10 values in Column "IMR32":
ENSG00000228505    113.166180
ENSG00000236605     87.690173
PHOX2B-AS1          84.152938
PLSCR5              76.831996
LINC01797           74.822022
LINC00682           74.017808
ENSG00000249122     73.515869
PHOX2B              70.710020
LINC01829           66.727985
ENSG00000251213     65.768828
Name: IMR32, dtype: float64

Gene descriptions for Column "IMR32":
novel transcript
novel transcript
PHOX2B antisense RNA 1
phospholipid scramblase family member 5
long intergenic non-protein coding RNA 1797
long intergenic non-protein coding RNA 682
novel transcript
paired like homeobox 2B
long intergenic non-protein coding RNA 1829
novel transcript

Top 10 values in Column "SKNSH":
ENSG00000213328    63.397441
PCDHA8             55.683877
TP53TG3HP          53.908791
PHOX2A             51.427785
MFFP3              49.528555
ENSG00000278009    47.338010
ENSG00000279239    43.861289
DBH                43.680099
RN7SL390P          42.273823
LINC02382          

Top 10 values in Column "Trachea":
SRGAP3-AS2         56.338440
RNU6-955P          55.785197
CD200R1L-AS1       54.915783
RNA5SP442          53.455672
STOML3             48.535802
ANKRD66            43.528209
SCGB3A1            39.936848
WFDC6              39.902812
ENSG00000236164    39.715075
ENSG00000261583    39.663310
Name: Trachea, dtype: float64

Gene descriptions for Column "Trachea":
SRGAP3 antisense RNA 2
RNA, U6 small nuclear 955, pseudogene
CD200R1L antisense RNA 1
RNA, 5S ribosomal pseudogene 442
stomatin like 3
ankyrin repeat domain 66
secretoglobin family 3A member 1
WAP four-disulfide core domain 6
novel transcript
novel transcript

Top 10 values in Column "Breast":
ENSG00000255627    50.382385
ENSG00000288671    41.726616
ENSG00000234918    36.483111
ANKRD30A           34.591417
ENSG00000254609    34.256830
IFNA7              33.917336
TMEM161BP1         32.521104
OR10J3             31.787600
VN1R53P            31.685018
RN7SL314P          29.914494
Name: Breast, dtype

Top 10 values in Column "SKOV3":
RNA5SP414    73.082549
SNORD18C     71.175972
RNA5SP413    70.401509
LINC02167    70.287552
VTRNA3-1P    67.233688
MIR10A       66.668840
SNORD138     64.429863
CTXND2       64.174846
SNORD63B     61.926130
SNORD36B     61.836119
Name: SKOV3, dtype: float64

Gene descriptions for Column "SKOV3":
RNA, 5S ribosomal pseudogene 414
small nucleolar RNA, C/D box 18C
RNA, 5S ribosomal pseudogene 413
long intergenic non-protein coding RNA 2167
vault RNA 3-1, pseudogene
microRNA 10a
small nucleolar RNA, C/D box 138
cortexin domain containing 2
small nucleolar RNA, C/D box 63B
small nucleolar RNA, C/D box 36B

Top 10 values in Column "Testis":
PRM1      111.408489
PRM2      111.018454
HMGB4     109.088683
SMCP      108.783036
BOD1L2    108.085947
AKAP4     106.810795
CETN1     106.801767
LELP1     106.465204
ACTRT2    104.704961
SCP2D1    104.387484
Name: Testis, dtype: float64

Gene descriptions for Column "Testis":
protamine 1
protamine 2
high mobility group bo

In [38]:
df = pd.DataFrame(low_cat_log2_oe_matrix, index=gene_names)
df.columns = category_list

for col in df.columns:
    top_10 = df[col].nlargest(10)
    print('Top 10 values in Column "{}":'.format(col))
    print(top_10)
    print()
    print('Gene descriptions for Column "{}":'.format(col))
    
    for gene in top_10.index:
        index = gene_names.index(gene)
        print(gene_description[index])
    print()


Top 10 values in Column "Heart":
RN7SKP288          5.907271
CACNA1C-IT1        5.888424
ENSG00000286639    5.634871
ENSG00000267784    5.631258
BTF3L4P4           5.521224
ENSG00000232056    5.499700
ENSG00000238156    5.445967
ENSG00000250500    5.412281
ENSG00000271401    5.217601
ENSG00000237324    5.191716
Name: Heart, dtype: float64

Gene descriptions for Column "Heart":
RN7SK pseudogene 288
CACNA1C intronic transcript 1
novel transcript
novel transcript, antisense to titin
basic transcription factor 3 like 4 pseudogene 4
novel transcript
novel transcript
eukaryotic translation initiation factor 2, subunit 3 gamma, 52kDa (EIF2S3) pseudogene
novel transcript
novel transcript

Top 10 values in Column "Atrium":
BMP10              6.531578
MIR208A            6.423206
ENSG00000287891    6.074370
MYBPHL             6.059309
ENSG00000261606    6.048335
ENSG00000225519    6.004728
LMAN1L             5.917669
MYH6               5.836798
LINC01880          5.775577
LINC02248          5.713

Top 10 values in Column "Hepatic Stellate Cell":
ENSG00000284695    4.560269
MIR656             4.420706
SULT1E1            4.237213
MIR412             4.193240
MIR369             4.036974
MIR410             3.882497
MIR409             3.805671
MIR377             3.613549
ENSG00000228707    3.472687
COLEC11            3.446299
Name: Hepatic Stellate Cell, dtype: float64

Gene descriptions for Column "Hepatic Stellate Cell":
novel protein
microRNA 656
sulfotransferase family 1E member 1
microRNA 412
microRNA 369
microRNA 410
microRNA 409
microRNA 377
novel transcript
collectin subfamily member 11

Top 10 values in Column "Hepatocyte":
GAGE12C    5.609262
GAGE12D    5.609262
APCS       5.475839
HPX        5.316595
HAO1       5.274673
HABP2      5.274520
FGB        5.214161
ANGPTL3    5.198160
FGA        5.196592
LBP        5.190555
Name: Hepatocyte, dtype: float64

Gene descriptions for Column "Hepatocyte":
G antigen 12C
G antigen 12D
amyloid P component, serum
hemopexin
hydroxyacid oxid

Top 10 values in Column "Macrophage":
ACOD1              5.879717
ENSG00000225107    5.233440
CHIT1              4.925902
ADAMDEC1           4.880459
LINC01181          4.797850
ETV3L              4.769616
DCSTAMP            4.650105
GPR31              4.644675
ENSG00000270980    4.641590
LINC02244          4.629506
Name: Macrophage, dtype: float64

Gene descriptions for Column "Macrophage":
aconitate decarboxylase 1
novel transcript
chitinase 1
ADAM like decysin 1
long intergenic non-protein coding RNA 1181
ETS variant transcription factor 3 like
dendrocyte expressed seven transmembrane protein
G protein-coupled receptor 31
coiled-coil-helix-coiled-coil-helix domain containing 2 (CHCHD2) pseudogene
long intergenic non-protein coding RNA 2244

Top 10 values in Column "Macroglia":
A1BG        NaN
A1BG-AS1    NaN
A1CF        NaN
A2M         NaN
A2M-AS1     NaN
A2ML1       NaN
A2ML1-AS1   NaN
A2ML1-AS2   NaN
A2MP1       NaN
A3GALT2     NaN
Name: Macroglia, dtype: float64

Gene description

Top 10 values in Column "DAOY":
ENSG00000233887    6.569105
CT47A1             6.373856
DAZ4               6.336465
CT47A11            6.308736
CT47A6             6.259546
ENSG00000278646    6.081961
DAZ1               6.052257
DAZ3               5.809056
FMR1NB             5.702992
TEX13C             5.619672
Name: DAOY, dtype: float64

Gene descriptions for Column "DAOY":
endogenous retroviral family W, env(C7), member 2
cancer/testis antigen family 47 member A1
deleted in azoospermia 4
cancer/testis antigen family 47 member A11
cancer/testis antigen family 47 member A6
novel protein similar to cancer/testis antigen family 47, member A12 (CT47A12)
deleted in azoospermia 1
deleted in azoospermia 3
FMR1 neighbor
TEX13 family member C

Top 10 values in Column "IMR32":
ENSG00000228505    6.822299
ENSG00000236605    6.454343
PHOX2B-AS1         6.394942
PLSCR5             6.263635
LINC01797          6.225391
LINC00682          6.209801
ENSG00000249122    6.199984
PHOX2B             6.14384

Top 10 values in Column "NHBE":
BPIFA1             6.086508
BPIFA2             5.888089
ENSG00000253339    5.617979
MSMB               5.389458
ENSG00000286225    5.310217
BPIFB1             5.225584
ATP12A             5.164898
ENSG00000251076    5.077920
CXCL17             4.994080
SCGB3A1            4.849893
Name: NHBE, dtype: float64

Gene descriptions for Column "NHBE":
BPI fold containing family A member 1
BPI fold containing family A member 2
novel transcript
microseminoprotein beta
novel transcript
BPI fold containing family B member 1
ATPase H+/K+ transporting non-gastric alpha2 subunit
novel transcript
C-X-C motif chemokine ligand 17
secretoglobin family 3A member 1

Top 10 values in Column "Trachea":
SRGAP3-AS2         5.816048
RNU6-955P          5.801810
CD200R1L-AS1       5.779149
RNA5SP442          5.740271
STOML3             5.600977
ANKRD66            5.443879
SCGB3A1            5.319649
WFDC6              5.318418
ENSG00000236164    5.311615
ENSG00000261583    5.309733


Top 10 values in Column "ES2":
ENSG00000251460    5.580810
LNCPRESS1          5.002081
ENSG00000274652    4.906275
ZSCAN10            4.868596
OR7E43P            4.799125
ENSG00000282048    4.666903
LNCPRESS2          4.465245
ENSG00000254277    4.456527
ENSG00000260362    4.390330
ENSG00000204368    4.355068
Name: ES2, dtype: float64

Gene descriptions for Column "ES2":
novel transcript
lncRNA p53 regulated and ESC associated 1
novel transcript
zinc finger and SCAN domain containing 10
olfactory receptor family 7 subfamily E member 43 pseudogene
novel transcript, sense intronic to OLFML3
lncRNA p53 regulated and ESC associated 2
novel transcript
novel transcript
SSX2 pseudogene (psiSSX2)

Top 10 values in Column "SKOV3":
RNA5SP414    6.191455
SNORD18C     6.153318
RNA5SP413    6.137534
LINC02167    6.135197
VTRNA3-1P    6.071112
MIR10A       6.058941
SNORD138     6.009658
CTXND2       6.003936
SNORD63B     5.952476
SNORD36B     5.950378
Name: SKOV3, dtype: float64

Gene descriptions f

---

In [40]:

# Compute row-wise averages while skipping NaN values
row_averages = np.nanmean(high_cat_matrix, axis=1)

# Create new matrix where each value is divided by row average
high_cat_oe_matrix = np.divide(high_cat_matrix, row_averages[:, np.newaxis], 
                       out=np.zeros_like(high_cat_matrix), where=row_averages[:, np.newaxis]!=0)

print("High Cat matrix shape:")
print(high_cat_matrix.shape)
print(high_cat_matrix[0][:5])
print()
print("Average:", row_averages[0])
print()
print("High Cat OE matrix shape:")
print(high_cat_oe_matrix.shape)
print(high_cat_oe_matrix[0][:5])

High Cat matrix shape:
(57447, 9)
[0.03641399 0.12006714 0.4371865  0.13372832 0.09514402]

Average: 0.12443487557940484

High Cat OE matrix shape:
(57447, 9)
[0.29263489 0.96489946 3.51337598 1.07468517 0.76460899]


In [28]:
# from IPython.display import display

# df = pd.DataFrame(high_cat_oe_matrix, index=gene_names)
# df.columns = ["Cardiovascular System",  "Connective Tissue", "Digestive System", 
# "Immune System", "Integumentary System", "Muscular System", "Nervous System", 
# "Respiratory System", "Urogenital System"]
# display(df)

In [29]:
# loop through each column and get the top 10 highest values and their indexes

df = pd.DataFrame(high_cat_matrix, index=gene_names)
df.columns = ["Cardiovascular System",  "Connective Tissue", "Digestive System", 
"Immune System", "Integumentary System", "Muscular System", "Nervous System", 
"Respiratory System", "Urogenital System"]

for col in df.columns:
    top_10 = df[col].nlargest(10)
    print('Top 10 values in Column "{}":'.format(col))
    print(top_10)
    print('Indexes for Column "{}":'.format(col))
    print(top_10.index)
    print()

Top 10 values in Column "Cardiovascular System":
GAPDH     5.647761
TMSB4X    5.485484
AHNAK     5.433829
RPL41     5.426241
EEF1A1    4.878405
FTH1      4.813606
RPS18     4.804848
TPM1      4.730638
TMSB10    4.587425
RPL13A    4.565091
Name: Cardiovascular System, dtype: float64
Indexes for Column "Cardiovascular System":
Index(['GAPDH', 'TMSB4X', 'AHNAK', 'RPL41', 'EEF1A1', 'FTH1', 'RPS18', 'TPM1',
       'TMSB10', 'RPL13A'],
      dtype='object')

Top 10 values in Column "Connective Tissue":
RPS18     5.857983
RPL41     5.741125
FTL       5.589432
EEF1A1    5.426893
RPL13A    4.826380
RPS12     4.747020
FTH1      4.700924
RPS25     4.661744
RPL11     4.627384
RPL19     4.623405
Name: Connective Tissue, dtype: float64
Indexes for Column "Connective Tissue":
Index(['RPS18', 'RPL41', 'FTL', 'EEF1A1', 'RPL13A', 'RPS12', 'FTH1', 'RPS25',
       'RPL11', 'RPL19'],
      dtype='object')

Top 10 values in Column "Digestive System":
FTL                5.594203
RPL41              5.491354
G

In [30]:
df = pd.DataFrame(high_cat_oe_matrix, index=gene_names)
df.columns = ["Cardiovascular System",  "Connective Tissue", "Digestive System", 
"Immune System", "Integumentary System", "Muscular System", "Nervous System", 
"Respiratory System", "Urogenital System"]

for col in df.columns:
    top_10 = df[col].nlargest(10)
    print('Top 10 values in Column "{}":'.format(col))
    print(top_10)
    print()
    print('Gene descriptions for Column "{}":'.format(col))
    
    for gene in top_10.index:
        index = gene_names.index(gene)
        print(gene_description[index])
    print()


Top 10 values in Column "Cardiovascular System":
MYBPC3             8.470539
ENSG00000236958    8.351298
MYL7               8.236753
ENSG00000286800    8.219700
ENSG00000287891    8.186218
NPPA               8.147898
LINC02502          8.103888
ENSG00000286820    8.073964
TNNI3K             7.764016
ENSG00000286598    7.692011
Name: Cardiovascular System, dtype: float64

Gene descriptions for Column "Cardiovascular System":
myosin binding protein C3
novel transcript
myosin light chain 7
novel transcript
novel transcript
natriuretic peptide A
long intergenic non-protein coding RNA 2502
novel transcript
TNNI3 interacting kinase
novel transcript

Top 10 values in Column "Connective Tissue":
LINC02573          8.243832
ENSG00000287973    8.194014
ENSG00000237311    8.117703
RHAG               8.083788
ABCB10P4           8.006390
HBBP1              7.996683
HBZ                7.980091
GATA1              7.937128
ENSG00000264334    7.889233
LINC01029          7.878699
Name: Connective Tissue

In [41]:
high_cat_log2_oe_matrix = np.log2(high_cat_oe_matrix)
df = pd.DataFrame(high_cat_log2_oe_matrix, index=gene_names)
df.columns = ["Cardiovascular System",  "Connective Tissue", "Digestive System", 
"Immune System", "Integumentary System", "Muscular System", "Nervous System", 
"Respiratory System", "Urogenital System"]

for col in df.columns:
    top_10 = df[col].nlargest(10)
    print('Top 10 values in Column "{}":'.format(col))
    print(top_10)
    print()
    print('Gene descriptions for Column "{}":'.format(col))
    
    for gene in top_10.index:
        index = gene_names.index(gene)
        print(gene_description[index])
    print()


Top 10 values in Column "Cardiovascular System":
MYBPC3             3.082454
ENSG00000236958    3.062000
MYL7               3.042076
ENSG00000286800    3.039086
ENSG00000287891    3.033197
NPPA               3.026428
LINC02502          3.018614
ENSG00000286820    3.013277
TNNI3K             2.956803
ENSG00000286598    2.943361
Name: Cardiovascular System, dtype: float64

Gene descriptions for Column "Cardiovascular System":
myosin binding protein C3
novel transcript
myosin light chain 7
novel transcript
novel transcript
natriuretic peptide A
long intergenic non-protein coding RNA 2502
novel transcript
TNNI3 interacting kinase
novel transcript

Top 10 values in Column "Connective Tissue":
LINC02573          3.043315
ENSG00000287973    3.034570
ENSG00000237311    3.021072
RHAG               3.015031
ABCB10P4           3.001152
HBBP1              2.999402
HBZ                2.996405
GATA1              2.988617
ENSG00000264334    2.979885
LINC01029          2.977957
Name: Connective Tissue

In [ ]:
Top 10 values in Column "Cardiovascular System":
MYBPC3             3.082454
ENSG00000236958    3.062000
MYL7               3.042076
ENSG00000286800    3.039086
ENSG00000287891    3.033197
NPPA               3.026428
LINC02502          3.018614
ENSG00000286820    3.013277
TNNI3K             2.956803
ENSG00000286598    2.943361
Name: Cardiovascular System, dtype: float64